<a href="https://colab.research.google.com/github/jiyanshud22/TEMP/blob/main/zoning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import random

# Sanofi H2O building data (from Excel, Scenario-1)
sanofi_data = {
    "Ground Floor": {
        "area_sqm": 1595,
        "categories": {
            "Task": 690.36,  # Individual + Interactive (Workstations, Focus, Touchdown)
            "Specialty": 450,
            "Support": 109.46,
            "Collaboration": 295.87,
            "Social": 49.31,
            "Entry": 0  # Assumed for client-facing/welcome
        }
    },
    "Level 1": {
        "area_sqm": 1181,
        "categories": {
            "Task": 272.21,
            "Specialty": 600,
            "Support": 172.68,
            "Collaboration": 116.66,
            "Social": 19.44,
            "Entry": 0
        }
    },
    "Level 2": {
        "area_sqm": 2363,
        "categories": {
            "Task": 1278.46,
            "Specialty": 250,
            "Support": 195.30,
            "Collaboration": 547.91,
            "Social": 91.32,
            "Entry": 0
        }
    },
    # Levels 3–6 same as Level 2
    "Level 3": {"area_sqm": 2363, "categories": {"Task": 1278.46, "Specialty": 250, "Support": 195.30, "Collaboration": 547.91, "Social": 91.32, "Entry": 0}},
    "Level 4": {"area_sqm": 2363, "categories": {"Task": 1278.46, "Specialty": 250, "Support": 195.30, "Collaboration": 547.91, "Social": 91.32, "Entry": 0}},
    "Level 5": {"area_sqm": 2363, "categories": {"Task": 1278.46, "Specialty": 250, "Support": 195.30, "Collaboration": 547.91, "Social": 91.32, "Entry": 0}},
    "Level 6": {"area_sqm": 2363, "categories": {"Task": 1278.46, "Specialty": 250, "Support": 195.30, "Collaboration": 547.91, "Social": 91.32, "Entry": 0}},
    "Level 7": {
        "area_sqm": 2327,
        "categories": {
            "Task": 690.36,
            "Specialty": 1100,
            "Support": 191.46,
            "Collaboration": 295.87,
            "Social": 49.31,
            "Entry": 0
        }
    },
    "Level 8": {
        "area_sqm": 2281,
        "categories": {
            "Task": 690.36,
            "Specialty": 1100,
            "Support": 145.46,
            "Collaboration": 295.87,
            "Social": 49.31,
            "Entry": 0
        }
    }
}

def calculate_default_counts(floor_data, rows, cols):
    total_cells = rows * cols
    area_sqm = floor_data["area_sqm"]
    sqm_per_cell = area_sqm / total_cells

    # Approximate Task as Workstations (80%) + Focus (20%)
    task_sqm = floor_data["categories"]["Task"]
    workstation_sqm = task_sqm * 0.8
    focus_sqm = task_sqm * 0.2

    # Convert SQM to cells
    counts = {
        "I": round(workstation_sqm / sqm_per_cell),  # Workstations
        "F": round(focus_sqm / sqm_per_cell),       # Focus Rooms/Phone Booths
        "C": round(floor_data["categories"]["Collaboration"] / sqm_per_cell),  # Collaboration
        "S": round(floor_data["categories"]["Support"] / sqm_per_cell),        # Support
        "O": round(floor_data["categories"]["Social"] / sqm_per_cell),         # Social
        "P": round(floor_data["categories"]["Specialty"] / sqm_per_cell),      # Specialty
        "E": round(floor_data["categories"]["Entry"] / sqm_per_cell)           # Entry
    }

    # Adjust counts to match total_cells
    total_count = sum(counts.values())
    if total_count != total_cells:
        # Scale proportionally
        factor = total_cells / total_count
        for key in counts:
            counts[key] = round(counts[key] * factor)
        # Fine-tune to exact total
        diff = total_cells - sum(counts.values())
        if diff > 0:
            counts["I"] += diff  # Add to workstations
        elif diff < 0:
            counts["I"] = max(0, counts["I"] + diff)  # Subtract from workstations

    return counts

def fill_zoning_grid(rows, cols, space_counts, periphery_cats=None, core_cats=None, entry_cats=None):
    total_cells = rows * cols
    if sum(space_counts.values()) != total_cells:
        print(f"Error: Sum of space categories must equal {total_cells}. Got {sum(space_counts.values())}.")
        return None

    # Initialize grid
    grid = [[None] * cols for _ in range(rows)]

    # Define positions
    # Periphery: Top, right, bottom, left edges
    periphery_positions = []
    for j in range(cols):
        periphery_positions.append((0, j))  # Top
    for i in range(1, rows):
        periphery_positions.append((i, cols - 1))  # Right
    for j in range(cols - 2, -1, -1):
        periphery_positions.append((rows - 1, j))  # Bottom
    for i in range(rows - 2, 0, -1):
        periphery_positions.append((i, 0))  # Left

    # Core: Center cells (approximate)
    core_rows = max(1, rows // 3)
    core_cols = max(1, cols // 3)
    core_start_row = (rows - core_rows) // 2
    core_start_col = (cols - core_cols) // 2
    core_positions = [(i, j) for i in range(core_start_row, core_start_row + core_rows)
                      for j in range(core_start_col, core_start_col + core_cols)]

    # Entry: Top-left corner
    entry_positions = [(0, 0), (0, 1), (1, 0), (1, 1)][:min(4, total_cells)]

    # Hub: Central but not core
    hub_positions = [(i, j) for i in range(1, rows - 1) for j in range(1, cols - 1)
                     if (i, j) not in core_positions]

    # Quiet zones: Non-periphery, non-entry, non-core
    quiet_positions = [(i, j) for i in range(1, rows - 1) for j in range(1, cols - 1)
                      if (i, j) not in core_positions and (i, j) not in hub_positions]

    # Fill entry positions
    for pos in entry_positions:
        i, j = pos
        if entry_cats and space_counts.get('E', 0) > 0:
            grid[i][j] = 'E'
            space_counts['E'] -= 1

    # Fill core positions (Support)
    for pos in core_positions:
        i, j = pos
        if grid[i][j] is not None:
            continue
        if core_cats and space_counts.get(core_cats[0], 0) > 0:
            grid[i][j] = core_cats[0]
            space_counts[core_cats[0]] -= 1

    # Fill periphery positions (Workstations)
    periphery_index = 0
    periphery_cats = periphery_cats or []
    for pos in periphery_positions:
        i, j = pos
        if grid[i][j] is not None:
            continue
        cat = periphery_cats[periphery_index % len(periphery_cats)] if periphery_cats else 'I'
        if space_counts.get(cat, 0) > 0:
            grid[i][j] = cat
            space_counts[cat] -= 1
        periphery_index += 1

    # Fill hub positions (Collaboration)
    for pos in hub_positions:
        i, j = pos
        if grid[i][j] is not None:
            continue
        if space_counts.get('C', 0) > 0:
            grid[i][j] = 'C'
            space_counts['C'] -= 1

    # Fill quiet zones (Focus)
    for pos in quiet_positions:
        i, j = pos
        if grid[i][j] is not None:
            continue
        if space_counts.get('F', 0) > 0:
            grid[i][j] = 'F'
            space_counts['F'] -= 1

    # Fill remaining with Social and Specialty
    empty_positions = [(i, j) for i in range(rows) for j in range(cols) if grid[i][j] is None]
    random.shuffle(empty_positions)

    remaining_cats = []
    for cat, count in space_counts.items():
        remaining_cats.extend([cat] * count)
    random.shuffle(remaining_cats)

    for idx, (i, j) in enumerate(empty_positions):
        if idx < len(remaining_cats):
            grid[i][j] = remaining_cats[idx]

    return grid

# Main program
print("Available floors:", ", ".join(sanofi_data.keys()))
floor = input("Enter the floor to plan (e.g., Ground Floor, Level 2): ").strip()
if floor not in sanofi_data:
    print("Invalid floor. Using Level 2 as default.")
    floor = "Level 2"

rows = int(input("Enter number of rows for the grid: "))
cols = int(input("Enter number of columns for the grid: "))
total_cells = rows * cols

# Get default counts from Sanofi data
default_counts = calculate_default_counts(sanofi_data[floor], rows, cols)

# Display defaults and ask for overrides
print("\nDefault space category counts (based on Sanofi data):")
category_names = {
    "I": "Workstation",
    "F": "Focus",
    "C": "Collaboration",
    "S": "Support",
    "O": "Social",
    "P": "Specialty",
    "E": "Entry"
}
for code, name in category_names.items():
    print(f"{name} ({code}): {default_counts.get(code, 0)} cells")

use_defaults = input("\nUse default counts? (y/n): ").strip().lower() == 'y'

if use_defaults:
    space_counts = default_counts
else:
    print("\nEnter custom cell counts for each category (must sum to", total_cells, "):")
    space_counts = {}
    for code, name in category_names.items():
        count = int(input(f"Number of {name} ({code}) cells: "))
        space_counts[code] = count

    # Validate total
    if sum(space_counts.values()) != total_cells:
        print("Error: Total cell counts do not match grid size. Adjusting Workstations.")
        space_counts["I"] += total_cells - sum(space_counts.values())

# Define constraints
periphery_cats = ["I"]  # Workstations on periphery
core_cats = ["S"]      # Support in core
entry_cats = ["E"]     # Entry zone

# Generate grid
grid = fill_zoning_grid(rows, cols, space_counts, periphery_cats, core_cats, entry_cats)

# Print grid
print(f"\nZoning Plan for Sanofi H2O {floor} ({rows}x{cols} Grid):")
for row in grid:
    print(" ".join(f"{cell:<2}" for cell in row))

# Print legend
print("\nLegend:")
for code, name in category_names.items():
    print(f"{code}: {name}")

Available floors: Ground Floor, Level 1, Level 2, Level 3, Level 4, Level 5, Level 6, Level 7, Level 8
Enter the floor to plan (e.g., Ground Floor, Level 2): Level 2
Enter number of rows for the grid: 5
Enter number of columns for the grid: 5

Default space category counts (based on Sanofi data):
Workstation (I): 10 cells
Focus (F): 3 cells
Collaboration (C): 6 cells
Support (S): 2 cells
Social (O): 1 cells
Specialty (P): 3 cells
Entry (E): 0 cells

Use default counts? (y/n): n

Enter custom cell counts for each category (must sum to 25 ):
Number of Workstation (I) cells: 5
Number of Focus (F) cells: 8
Number of Collaboration (C) cells: 3
Number of Support (S) cells: 4
Number of Social (O) cells: 1
Number of Specialty (P) cells: 2
Number of Entry (E) cells: 2

Zoning Plan for Sanofi H2O Level 2 (5x5 Grid):
E  E  I  I  I 
F  C  C  C  I 
F  S  S  P  I 
P  F  S  F  F 
O  S  F  F  F 

Legend:
I: Workstation
F: Focus
C: Collaboration
S: Support
O: Social
P: Specialty
E: Entry
